![JupyterLite HIV Drug Resistance Maps: an original conceptual globe, map, and molecular illustration.](assets/hivdr-splash.png)

**[Open this notebook in JupyterLite Lab](https://jltobias.github.io/JupyterLite-HIVDB-Drug-Resistance-Maps/lab/index.html?path=12_Meta_Analysis.ipynb)** · **[Read the Jupyter Book](https://jltobias.github.io/JupyterLite-HIVDB-Drug-Resistance-Maps/book/)** · [Map and globe resources](mapping_resources.md)

Original conceptual artwork; its markers and bars are not observations.

# Meta Analysis

Project: JupyterLite HIV Drug Resistance Maps. Created/updated: 2026-11-16.
Author: [Repository contributors; generated with Codex]. Email/organization: [Not supplied].
Input: factual HIVDB metadata snapshot in `data/studies/`, plus the source-checked annotation ledger.
Output: computed pandas tables and Plotly figures; optional exports under `outputs/`.
Runtime: Python 3.11+, pandas, Plotly 6.2.0, IPython; native retrieval dependencies are listed in `study_query.py`.
Change log: initial reproducible study-inventory notebook. No simulated study data.

**Unit:** a distinct HIVDB reference/publication ID. A publication can describe multiple study components; multiple publications can describe one cohort. This is a descriptive inventory, not a population sample, national resistance estimate, or deduplicated systematic review.

## From the country matrix to research questions across Africa

This culmination uses the preceding matrix for **Africa-wide evidence mapping** and the harmonized [Rhee et al. (2015) S1 Table](https://doi.org/10.1371/journal.pmed.1001810.s004) for a separate, exploratory synthesis of **overall transmitted drug resistance (TDR) in ART-naive populations**, as requested.

**Working study specification.** The inventory unit is a publication-country association. The pooling unit is one eligible study row in the published harmonized dataset. Outcome: the proportion with at least one surveillance drug-resistance mutation in the sequenced pol regions, under the source study's harmonized definition. This is a single-arm prevalence synthesis, not a treatment comparison or network meta-analysis. Primary geography: single-country studies wholly within UN M49 Africa. Multicountry totals are not duplicated across countries. Publication year is not collection year. No participant-level weighting, survey weights or new causal model are assumed.

**Read this before the pooled result.** S1 supplies one-decimal percentages and participant counts, not a directly reported overall event-count column. We retain a count only when exactly one integer is compatible with that percentage and denominator, and it satisfies the bounds implied by mutation marginals. Counts remain **conditionally reconstructed**, not verified original numerators. Inconsistent rows remain in an exclusion audit. Summing mutation counts would double-count people and is never used as the numerator. Historical CPR report links did not respond during preparation.

The model is therefore a conditional research demonstration on a documented subset, not a validated estimate for all African countries, all HIVDB studies, or today's resistance. No independent risk-of-bias review or cross-publication cohort-overlap adjudication has been completed. Independent study effects are an explicit exploratory assumption; resolve it before a formal synthesis.

In [ ]:
# Read immutable factual inputs and check their recorded hashes before analysis.
from pathlib import Path
import json
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from IPython.display import display, Code, HTML, FileLink
from study_matrix import (load_snapshot, country_membership, build_domains,
                          country_matrix, coverage_table, country_summary,
                          evidence_ledger, DOMAINS, UNKNOWN)
catalogue, evidence, signals, annotations, manifest = load_snapshot()
px.defaults.template = "plotly_white"
print("Snapshot built:", manifest["built_utc"])
print("Catalogue publication records:", len(catalogue))

In [ ]:
# Read the independently audited harmonized data and factual Africa membership.
import numpy as np
from meta_analysis_tools import load_meta_inputs, study_effects, reml_meta, pool
from scipy.special import expit
from scipy.stats import norm
studies, africa_areas, meta_manifest = load_meta_inputs()
africa_iso3 = set(africa_areas.iso3)
africa_evidence = evidence.loc[evidence.iso3.isin(africa_iso3)]
africa_refs = set(africa_evidence.ref_id)
africa_catalogue = catalogue.loc[catalogue.ref_id.isin(africa_refs)]
africa_facts = build_domains(catalogue, evidence, annotations)
africa_facts = africa_facts.loc[africa_facts.iso3.isin(africa_iso3)]
print("African countries/areas in geographic frame:", len(africa_areas))
print("Countries/areas with selected-source evidence:", africa_evidence.iso3.nunique())
print("Distinct publication records with African sampling metadata:", len(africa_refs))
print("Metadata snapshot:", manifest["built_utc"])
print("Harmonized historical source: 2015 publication; see actual sampling years below.")

## 1. Explore the African study matrix

The map includes every M49 African country/area, including locations with no mapped publication in this snapshot. Zero here is a catalogue count, not a clinical result. Geographical groupings serve statistical description and do not imply a position on territorial status.

In [ ]:
# Join the same country summary used in Notebook 11 to the full Africa frame.
africa_summary = africa_areas.merge(country_summary(catalogue, evidence), on="iso3", how="left", validate="one_to_one")
count_cols = ["Publications","DTG query publications","LEN query publications","Publications with exact sample year"]
africa_summary[count_cols] = africa_summary[count_cols].fillna(0).astype(int)
fig = px.choropleth(africa_summary, locations="iso3", color="Publications", hover_name="country_area",
    hover_data=["subregion","DTG query publications","LEN query publications"], scope="africa",
    color_continuous_scale="Teal", title="African publication coverage in the study matrix")
fig.update_layout(height=560,margin=dict(l=0,r=0,t=55,b=0))
fig.show()
display(africa_summary[["country_area","subregion",*count_cols]].sort_values("Publications",ascending=False).head(15))

In [ ]:
# All requested domains, counted as availability of source-supported information.
known = africa_facts.loc[africa_facts.value.ne(UNKNOWN)]
africa_domain_matrix = known.pivot_table(index="country", columns="domain", values="ref_id", aggfunc="nunique", fill_value=0)
africa_domain_matrix = africa_domain_matrix.reindex(columns=list(DOMAINS),fill_value=0).astype(int)
top = africa_evidence.groupby("country").ref_id.nunique().nlargest(18).index
fig = px.imshow(africa_domain_matrix.reindex(top,fill_value=0), aspect="auto", color_continuous_scale="Blues",
    labels=dict(color="Publications",x="Domain",y="Country"), title="Where does domain extraction remain incomplete?")
fig.update_layout(height=590)
fig.show()

## 2. Link harmonized studies back to the catalogue

The link uses a nonempty PubMed ID, not similar author names. A source study without a matching catalogue PMID remains eligible for the historical analysis and is explicitly marked unlinked. Matching a publication does not prove that every sequence or denominator matches between database releases. The newer query subsets do not supply resistance prevalence denominators.

In [ ]:
# A PMID can map to several database RefIDs; preserve the full set without multiplying study rows.
eligible = studies.loc[studies.inclusion_reason.eq("Included (conditional)")].copy()
assert eligible.study_id.is_unique
assert eligible.loc[eligible.pmid.ne(""), "pmid"].is_unique
assert eligible.iso3.isin(africa_iso3).all()
print(f"Conditional pooling subset: {len(eligible)} studies, {eligible.iso3.nunique()} countries, N={eligible.n.sum():,}")
print("Median sampling-year range:", eligible.median_sample_year.min(), "to", eligible.median_sample_year.max())
print("Linked by PMID to the matrix:", eligible.matrix_ref_ids.ne("").sum(), "of", len(eligible))
display(eligible[["study_id","study_label","iso3","pmid","matrix_ref_ids","n","overall_tdr_pct_reported","events_reconstructed"]].head(15))

## 3. Eligibility and denominator audit

The flow below is a reanalysis eligibility flow, **not a new PRISMA systematic-search flow**. The original study selection is described in the 2015 article. No current whole-database search, full-text screening, or certainty-of-evidence grading is claimed. The source table's regional labels are checked against explicit country membership; for example, its Northern Africa group includes an Iranian study, which this analysis correctly excludes from Africa.

In [ ]:
# Every source row has exactly one primary inclusion/exclusion reason.
flow = studies.inclusion_reason.value_counts().rename_axis("Decision").reset_index(name="Studies")
assert flow.Studies.sum() == len(studies) == 287
display(flow)
fig = px.bar(flow, x="Studies", y="Decision", orientation="h", text="Studies",
    title="Audited selection from the 287-study harmonized source", color_discrete_sequence=["#087e8b"])
fig.update_layout(yaxis=dict(autorange="reversed"),height=350)
fig.show()
# Keep questionable input values visible rather than forcing event counts by rounding.
exclusions = studies.loc[studies.wholly_in_africa & ~studies.inclusion_reason.eq("Included (conditional)")]
display(exclusions[["study_id","study_label","countries_raw","n","overall_tdr_pct_reported","inclusion_reason"]])

## 4. Reproducible model and assumptions

For eligible study *i*, let `x` be the reconstructed count and `n` the reported denominator. For interior proportions, `y = log(x / (n − x))` and `v = 1/x + 1/(n − x)`. For zero-event or all-event studies only, add 0.5 to both event and nonevent cells. A sensitivity analysis changes this correction.

Fit an intercept-only random-effects normal model to study logits, estimating between-study variance **τ² by REML**. Use a modified Hartung–Knapp interval with a residual scale floor of 1 and `t(k−1)` quantiles. Report Q, I² and τ² as descriptive heterogeneity measures; none determines whether pooling is scientifically valid. The approximate prediction interval uses `t(k−2)` and includes between-study variance. τ² uncertainty is not fully propagated, so the prediction interval is approximate, especially in small subgroups.

Back-transforming the mean logit describes a **typical included-study proportion** under this model. It is not the arithmetic population prevalence of Africa. Unequal surveillance coverage, changing calendar periods, selection bias, reconstructed numerators and dependence can all affect it. See [meta_analysis_tools.py](meta_analysis_tools.py) for the annotated objective and numerical checks.

In [ ]:
# The exact computed effects feed all tables, forest plots and sensitivity analyses.
effects, result = pool(eligible, correction=0.5)
summary_table = pd.DataFrame([{
    "Studies": result["k"], "Countries": eligible.iso3.nunique(), "Participants (reported N)": int(eligible.n.sum()),
    "Typical study TDR (%)": 100*result["pooled"],
    "95% CI lower (%)": 100*result["ci_lower"], "95% CI upper (%)": 100*result["ci_upper"],
    "Approx. prediction lower (%)": 100*result["pi_lower"], "Approx. prediction upper (%)": 100*result["pi_upper"],
    "Q": result["Q"], "Q df": result["Q_df"], "I-squared (%)": result["I2_percent"], "Tau-squared (logit scale)": result["tau2"]
}])
display(summary_table.style.format(precision=3).set_caption("CONDITIONAL exploratory synthesis — reconstructed event counts"))
print("Boundary studies receiving a continuity correction:", int(effects.continuity_corrected.sum()))
print("This result describes the selected historical studies; it is not current national or continental prevalence.")

## 5. Forest plot of every included study

Points and intervals use the model's logit scale before back-transformation. Marker area reflects random-effects weight. For corrected zero-event studies, the model point is above zero; hover shows the original reported percentage. The pooled interval and prediction interval describe different uncertainties.

In [ ]:
# Sort for reading only; preserve stable study IDs in every exported result.
plot_data = effects.sort_values(["iso3","median_sample_year","study_id"]).reset_index(drop=True)
labels = plot_data.iso3 + " | " + plot_data.study_label + " | " + plot_data.study_id
forest = go.Figure(go.Scatter(x=100*plot_data.study_effect_proportion, y=labels, mode="markers",
    marker=dict(size=plot_data.weight_pct, sizemode="area", sizeref=2*plot_data.weight_pct.max()/18**2,
                sizemin=4, color="#087e8b",symbol="square"),
    error_x=dict(type="data",symmetric=False,array=100*(plot_data.study_upper-plot_data.study_effect_proportion),
                 arrayminus=100*(plot_data.study_effect_proportion-plot_data.study_lower)),
    customdata=plot_data[["n","events_reconstructed","overall_tdr_pct_reported","weight_pct"]].to_numpy(),
    hovertemplate="%{y}<br>Model point: %{x:.2f}%<br>N: %{customdata[0]}<br>Reconstructed events: %{customdata[1]}<br>Reported: %{customdata[2]}%<br>Weight: %{customdata[3]:.2f}%<extra></extra>",name="Included studies"))
forest.add_trace(go.Scatter(x=[100*result["pooled"]],y=["REML typical study (conditional)"],mode="markers",
    marker=dict(symbol="diamond",size=14,color="#bb4b38"),
    error_x=dict(type="data",symmetric=False,array=[100*(result["ci_upper"]-result["pooled"])],
                 arrayminus=[100*(result["pooled"]-result["ci_lower"])]),name="Pooled 95% CI"))
forest.add_trace(go.Scatter(x=[100*result["pi_lower"],100*result["pi_upper"]], y=["Approximate prediction interval"]*2,
    mode="lines",line=dict(color="#bb4b38",width=4,dash="dot"),name="Prediction interval"))
forest.update_layout(title="Overall TDR in ART-naive populations — eligible African studies (historical)",
    xaxis_title="Model proportion with overall TDR (%)", yaxis=dict(autorange="reversed"),
    height=max(600,24*len(effects)+220), margin=dict(l=330,r=20,t=80,b=60),legend=dict(orientation="h",y=1.02))
forest.show()

## 6. Country subgroups and collection periods

Country summaries are shown only when at least three eligible studies exist, a display rule for this exploratory notebook rather than a universal sufficiency criterion. Even three studies can leave extreme uncertainty. These are within-country study syntheses, not nationally representative estimates. Cross-country differences and collection-year patterns do not establish causal effects of policies or drug rollout.

In [ ]:
country_results = []
for iso, group in eligible.groupby("iso3"):
    if len(group) < 3:
        continue
    _, r = pool(group)
    country_results.append({"iso3":iso,"Studies":len(group),"Typical study TDR (%)":100*r["pooled"],
        "CI lower (%)":100*r["ci_lower"],"CI upper (%)":100*r["ci_upper"],"I2 (%)":r["I2_percent"],
        "Sample year min":group.median_sample_year.min(),"Sample year max":group.median_sample_year.max()})
country_results = pd.DataFrame(country_results).merge(africa_areas,on="iso3",validate="one_to_one")
display(country_results[["country_area","Studies","Typical study TDR (%)","CI lower (%)","CI upper (%)","I2 (%)","Sample year min","Sample year max"]].style.format(precision=2))
fig = px.scatter(effects,x="median_sample_year",y="overall_tdr_pct_reported",size="n",color="iso3",
    hover_name="study_label", labels={"median_sample_year":"Median collection year","overall_tdr_pct_reported":"Reported overall TDR (%)"},
    title="Historical study estimates by collection year — descriptive, not a fitted time trend")
fig.update_layout(height=470,showlegend=False)
fig.show()

## 7. Sensitivity, influence and missing evidence

Check continuity-correction choices, restrict to sequentially sampled studies, and leave out one country at a time. These checks show model sensitivity; they do not repair outcome-dependent exclusions, cohort overlap or selection bias. Funnel asymmetry is not a diagnostic test of publication bias here: prevalence and its estimated standard error are mechanically related, and populations and sampling periods differ. No automatic Egger test or bias “correction” is applied.

In [ ]:
sensitivity = []
for label, data, correction in [
    ("Primary: boundary correction 0.5",eligible,.5),
    ("Boundary correction 0.25",eligible,.25),
    ("Boundary correction 1.0",eligible,1.0),
    ("Sequential sampling only",eligible.loc[eligible.sampling_method.eq("Sequential Samples")],.5),
    ("Records with a PubMed identifier",eligible.loc[eligible.pmid.ne("")],.5),
]:
    if len(data) < 3:
        continue
    _, r = pool(data,correction=correction)
    sensitivity.append({"Analysis":label,"Studies":len(data),"TDR (%)":100*r["pooled"],
                        "CI lower (%)":100*r["ci_lower"],"CI upper (%)":100*r["ci_upper"]})
display(pd.DataFrame(sensitivity).style.format(precision=3))
influence = []
for iso in sorted(eligible.iso3.unique()):
    _, r = pool(eligible.loc[eligible.iso3.ne(iso)])
    influence.append({"Country omitted":iso,"Typical TDR (%)":100*r["pooled"]})
influence = pd.DataFrame(influence)
fig = px.scatter(influence,x="Typical TDR (%)",y="Country omitted",title="Leave-one-country-out sensitivity")
fig.add_vline(x=100*result["pooled"],line_dash="dash",line_color="#bb4b38")
fig.update_layout(height=630)
fig.show()

In [ ]:
# A descriptive funnel plot; no claim that its shape identifies reporting bias.
funnel = px.scatter(effects,x="logit",y="se",hover_name="study_label",color="iso3",
    title="Descriptive funnel: study logit and approximate standard error",
    labels={"logit":"Logit TDR (boundary-corrected when needed)","se":"Approximate standard error"})
funnel.add_vline(x=result["mu_logit"],line_dash="dash",line_color="#bb4b38")
funnel.update_yaxes(autorange="reversed")
funnel.update_layout(height=450,showlegend=False)
funnel.show()

## 8. Research questions supported by the evidence map

These questions are generated from actual source coverage and analysis inputs. “Supported for screening” does not mean sufficient for a pooled estimate. The broader catalogue is useful for finding studies; a current clinical synthesis still needs full-text outcome extraction, denominator reconciliation, cohort linkage and risk-of-bias assessment.

In [ ]:
# Distinct reference counts; never sum publication-country associations as studies.
dtg_refs = africa_evidence.loc[africa_evidence.source.eq("Live DTG query"),"ref_id"].nunique()
len_refs = africa_evidence.loc[africa_evidence.source.eq("Live LEN query"),"ref_id"].nunique()
pediatric_refs = africa_facts.loc[africa_facts.domain.eq("age_group") & africa_facts.value.eq("Children / adolescents"),"ref_id"].nunique()
phenotype_refs = africa_catalogue.loc[africa_catalogue.phenotype_reference.eq("True"),"ref_id"].nunique()
questions = pd.DataFrame([
    ["How does historical TDR vary by collection period and recruitment setting?",
     f"{len(eligible)} eligible study rows; sample years {eligible.median_sample_year.min()}–{eligible.median_sample_year.max()}.",
     "Exploratory descriptive comparison available; prespecify a small number of moderators and reconcile excluded counts before meta-regression."],
    ["Which African countries have sparse documented sampling and poor domain coverage?",
     f"{len(africa_refs)} publication IDs across {africa_evidence.iso3.nunique()} mapped countries/areas.",
     "Evidence-gap mapping is available now; low catalogue counts do not measure low disease burden."],
    ["What is DTG resistance among people with virological failure, by prior ART and age?",
     f"{dtg_refs} publication IDs with African sampling in the explicit DTG exposure query.",
     "Screen these references; extract the tested failure denominator, resistance definition, age and prior ART. Do not pool query isolate counts as prevalence."],
    ["Where are pediatric and adolescent resistance outcomes insufficiently characterized?",
     f"{pediatric_refs} African publication IDs have a source-checked pediatric annotation in the small worked set.",
     "Expand full-text age and population extraction; current missing age fields preclude a complete pediatric comparison."],
    ["How do LEN resistance questions differ between prevention, initial therapy and salvage therapy?",
     f"{len_refs} publication IDs with African sampling in the explicit LEN exposure query.",
     "Keep prevention and treatment estimands separate; catalogue/query coverage alone cannot establish effectiveness or resistance risk."],
    ["Do genotype findings have corresponding phenotypic susceptibility evidence?",
     f"{phenotype_refs} African-associated reference IDs also occur in the phenotype reference export.",
     "Link exact isolate/assay records before interpreting concordance; a shared publication ID does not locate laboratory experiments in a country."],
],columns=["Research question","Observed support","Next analytical step"])
display(questions.style.hide(axis="index").set_properties(**{"text-align":"left"}))

## 9. Export and extend the synthesis

The export includes every eligible effect, exclusions, a separate screening template, conditional model summary, country subgroups, influence checks and research questions. The screening template starts with “Not assessed” rather than fabricated risk-of-bias ratings. For a formal review, define and register an appropriate protocol, verify all outcome denominators and cohort independence, and use PRISMA 2020 and an appropriate certainty-of-evidence framework. These are future research steps, not claims about the present exploratory notebook.

In [ ]:
out = Path("outputs/meta_analysis"); out.mkdir(parents=True,exist_ok=True)
effects.to_csv(out / "conditional_study_effects.csv",index=False)
exclusions.to_csv(out / "exclusion_audit.csv",index=False)
summary_table.to_csv(out / "conditional_summary.csv",index=False)
summary_table.style.format(precision=4).to_html(out / "conditional_summary.html")
country_results.to_csv(out / "country_subgroups.csv",index=False)
influence.to_csv(out / "leave_one_country_out.csv",index=False)
questions.to_csv(out / "research_questions.csv",index=False)
africa_domain_matrix.to_csv(out / "africa_domain_matrix.csv")
screening = studies.loc[studies.wholly_in_africa,["study_id","study_label","pmid","countries_raw","inclusion_reason"]].copy()
for field in ["denominator_verified","cohort_overlap_review","sampling_bias","assay_definition","risk_of_bias"]:
    screening[field] = "Not assessed"
screening.to_csv(out / "screening_template.csv",index=False)
forest.write_html(out / "forest.html",include_plotlyjs=True)
funnel.write_html(out / "funnel.html",include_plotlyjs=True)
print("Created:", ", ".join(p.name for p in sorted(out.iterdir())))

## Sources and methods

- [Rhee et al. (2015), PLOS Medicine](https://doi.org/10.1371/journal.pmed.1001810) and [CC0 S1 Table](https://doi.org/10.1371/journal.pmed.1001810.s004): harmonized historical input. This notebook is a new subset analysis, not a reproduction of the paper's participant-level models.
- [UN Statistics Division M49](https://unstats.un.org/unsd/methodology/m49/): factual country/area and subregion membership. The source regional label is retained for auditing.
- [Cochrane Handbook, Chapter 10](https://www.cochrane.org/authors/handbooks-and-manuals/handbook/current/chapter-10): effect synthesis, heterogeneity and sensitivity; [Chapter 13](https://www.cochrane.org/authors/handbooks-and-manuals/handbook/current/chapter-13): missing-evidence considerations.
- [Viechtbauer (2010)](https://doi.org/10.18637/jss.v036.i03): random/mixed-effects meta-analysis methods. [Röver, Knapp and Friede (2015)](https://doi.org/10.1186/s12874-015-0091-1): modified Hartung–Knapp inference.
- [SciPy scalar optimization](https://docs.scipy.org/doc/scipy/reference/generated/scipy.optimize.minimize_scalar.html): numerical REML implementation. [PRISMA 2020](https://www.prisma-statement.org/prisma-2020): reporting for a future formal systematic review.

All graphics here are generated from the computed tables with Plotly. No publication figures, abstracts, logos or screenshots are reproduced. [Source and field documentation](study_matrix_methods.md) describes the larger study matrix and its limitations.